# RAG visual: buscar documentos por lo que se VE

**Lección 6 · Clase 5.3** — recuperar la página correcta de un PDF **sin extraer texto**: cada página se indexa **como imagen** con un embedding multimodal, y la pregunta la encuentra por similitud.

¿Por qué? La mayoría de los documentos de una empresa son **visuales**: facturas con timbres, contratos con firmas, informes llenos de tablas y gráficos, presentaciones. El camino clásico (OCR → texto → RAG) pierde justo lo que importa: el layout de la tabla, el gráfico completo, el sello.

| | RAG clásico (OCR/parsing) | RAG visual |
|---|---|---|
| Preproceso | OCR + chunking (frágil, lento) | Rasterizar la página y listo |
| Tablas y gráficos | Se degradan o se pierden | Quedan intactos: se indexa el píxel |
| Qué se recupera | Fragmentos de texto | **La página completa, como imagen** |
| Quién responde | LLM leyendo texto | LLM multimodal **mirando** la página |

La idea viene de **ColPali** (y su sucesor ColQwen): modelos que embeben la página-imagen directamente y dominan **ViDoRe**, el benchmark de retrieval sobre documentos visuales. La versión open-source (`colpali-engine`) exige GPU para ser práctica; en esta lección usamos el mismo patrón en su **versión API** — `gemini-embedding-2`, el embedding multimodal de la Gemini API — demostrable en vivo desde un laptop.

In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q google-genai==2.16.0 pypdfium2==5.12.1 numpy==2.5.1 pillow==12.3.0 langchain-openai==1.3.5 langchain-core==1.4.9 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

# Carga OPENAI_API_KEY y GEMINI_API_KEY desde .env si existe (local); en Colab usa Secrets.
load_dotenv()

try:
    from google.colab import userdata  # type: ignore
    for llave in ("OPENAI_API_KEY", "GEMINI_API_KEY"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
HAY_GEMINI = bool(os.environ.get("GEMINI_API_KEY"))
print("OPENAI_API_KEY presente:", HAY_OPENAI)
print("GEMINI_API_KEY presente:", HAY_GEMINI)
if not HAY_GEMINI:
    print("⚠️ crea una API key gratis en aistudio.google.com y ponla en .env como GEMINI_API_KEY")


## El documento

`data/documento_ejemplo.pdf` — extracto del capítulo II del **IPoM de junio 2026** del Banco Central de Chile (*Evolución futura de la política monetaria*, págs. 35-43 del original, 9 páginas). Es el documento perfecto para esta lección: texto denso, 4 tablas y 7 gráficos.

> Fuente: Banco Central de Chile, Informe de Política Monetaria (IPoM), junio 2026 — https://www.bcentral.cl/publicaciones/politicas/informe-de-politica-monetaria-ipom (documento público; extracto preparado en julio 2026). Es **el mismo PDF de la lección 5**: allá lo *parseamos* a texto, acá lo buscamos **sin parsearlo**.

El único preproceso del RAG visual: convertir cada página a imagen. `pypdfium2` (el motor de PDF de Chrome, empaquetado para Python) rasteriza a la resolución que pidamos — sin OCR, sin detectar tablas, sin chunking.

In [ ]:
from pathlib import Path

import pypdfium2 as pdfium
from IPython.display import display

pdf_path = Path("data/documento_ejemplo.pdf")
if not pdf_path.exists():
    import urllib.request
    # En Colab el archivo local no existe: se baja desde el repo (público)
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(
        "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
        "class_5_3_imagenes/leccion6_rag_visual/data/documento_ejemplo.pdf",
        headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"},
    )
    pdf_path.write_bytes(urllib.request.urlopen(req).read())

pdf = pdfium.PdfDocument(pdf_path)
paginas = [page.render(scale=2.0).to_pil() for page in pdf]  # scale=2.0 ≈ 144 dpi
pdf.close()
print(f"{len(paginas)} páginas rasterizadas, tamaño {paginas[0].size} px")

for n in (1, 2, 6):  # miniaturas: portada, la tabla de supuestos y los gráficos de actividad
    print(f"--- página {n} ---")
    display(paginas[n - 1].reduce(4))


## Indexar: un embedding por página-imagen

Cada página viaja a `gemini-embedding-2` — el primer embedding **multimodal** de la Gemini API — y vuelve como un vector que captura texto, tablas y gráficos a la vez. La firma exacta viene de la [documentación oficial de embeddings](https://ai.google.dev/gemini-api/docs/embeddings):

- Cada imagen va como `types.Part.from_bytes(data=..., mime_type="image/png")`, envuelta en su propio `types.Content` para recibir **un embedding por página** (sin envolver, la API devuelve un solo embedding agregado).
- Máximo **6 imágenes por request** → indexamos por lotes.
- `output_dimensionality` es flexible (128–3072); usamos 1536 y normalizamos los vectores (con eso, similitud coseno = producto punto).

El índice completo es una **matriz numpy** — para 9 páginas no hace falta ninguna base de datos vectorial.

In [ ]:
import io

import numpy as np

indice = None  # matriz (páginas × dimensiones); queda en None si no hay llave
DIMENSIONES = 1536


def png_bytes(imagen) -> bytes:
    buf = io.BytesIO()
    imagen.save(buf, format="PNG")
    return buf.getvalue()


print(f"Costo estimado del indexado: {len(paginas)} páginas × US$0.00012 = US${len(paginas) * 0.00012:.5f}")

if not HAY_GEMINI:
    print("⚠️ Falta GEMINI_API_KEY — crea una API key gratis en aistudio.google.com y ponla en .env como GEMINI_API_KEY.")
    print("Se salta el indexado (y las celdas que dependen de él).")
else:
    from google import genai
    from google.genai import types

    client = genai.Client()  # lee GEMINI_API_KEY del entorno

    LOTE = 6  # máximo de imágenes por request según la doc oficial
    vectores = []
    for inicio in range(0, len(paginas), LOTE):
        respuesta = client.models.embed_content(
            model="gemini-embedding-2",
            contents=[
                types.Content(parts=[types.Part.from_bytes(data=png_bytes(img), mime_type="image/png")])
                for img in paginas[inicio : inicio + LOTE]
            ],
            config=types.EmbedContentConfig(output_dimensionality=DIMENSIONES),
        )
        vectores.extend(e.values for e in respuesta.embeddings)

    indice = np.array(vectores)
    indice = indice / np.linalg.norm(indice, axis=1, keepdims=True)  # normalizar → coseno = producto punto
    print(f"Índice listo: matriz de {indice.shape[0]} páginas × {indice.shape[1]} dimensiones")


## Consultar: la pregunta encuentra la página

La pregunta (texto) se embebe en el **mismo espacio vectorial** que las páginas (imágenes) — esa es la gracia del embedding multimodal. Según la doc oficial, para retrieval la instrucción de tarea va dentro del prompt: `task: search result | query: ...`.

La búsqueda es una línea de numpy: producto punto entre la matriz del índice y el vector de la pregunta = similitud coseno con las 9 páginas de una vez.

In [ ]:
def buscar_paginas(pregunta: str, k: int = 3) -> list[tuple[int, float]]:
    """Devuelve las k páginas más parecidas a la pregunta: [(número_de_página, similitud), ...]"""
    respuesta = client.models.embed_content(
        model="gemini-embedding-2",
        contents=f"task: search result | query: {pregunta}",
        config=types.EmbedContentConfig(output_dimensionality=DIMENSIONES),
    )
    q = np.array(respuesta.embeddings[0].values)
    q = q / np.linalg.norm(q)
    similitudes = indice @ q  # similitud coseno contra todas las páginas a la vez
    top = np.argsort(similitudes)[::-1][:k]
    return [(int(i) + 1, float(similitudes[i])) for i in top]


if indice is None:
    print("⚠️ No hay índice (falta GEMINI_API_KEY) — esta celda se salta.")
else:
    pregunta = "¿Cuál es el precio del cobre proyectado para 2026, 2027 y 2028?"
    print(f"Pregunta: {pregunta}\n")
    for numero, similitud in buscar_paginas(pregunta):
        print(f"Página {numero} — similitud coseno {similitud:.3f}")
        display(paginas[numero - 1].reduce(6))


## Responder: la página recuperada entra como imagen a GPT-5

Igual que en la **lección 2**: la página top-1 viaja en base64 dentro de un `HumanMessage`, junto a la pregunta. El modelo responde **mirando la página** — tabla y gráficos incluidos — y cita el número de página, porque el retrieval nos dijo cuál es.

In [ ]:
import base64

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI


def responder(pregunta: str) -> str:
    numero, similitud = buscar_paginas(pregunta, k=1)[0]
    print(f"→ página recuperada: {numero} (similitud {similitud:.3f})")
    mensaje = HumanMessage(
        content=[
            {
                "type": "text",
                "text": (
                    f"Esta imagen es la página {numero} de un informe del Banco Central de Chile. "
                    f"Responde en español, breve y solo con lo que se ve en la página: {pregunta} "
                    f"Termina con la cita '(fuente: página {numero})'."
                ),
            },
            {
                "type": "image",
                "source_type": "base64",
                "data": base64.b64encode(png_bytes(paginas[numero - 1])).decode("utf-8"),
                "mime_type": "image/png",
            },
        ]
    )
    return llm.invoke([mensaje]).content


if indice is None or not HAY_OPENAI:
    print("⚠️ Falta el índice (GEMINI_API_KEY) o la llave de OpenAI — esta celda se salta.")
else:
    llm = ChatOpenAI(model="gpt-5")
    print(responder("¿Cuál es el precio del cobre proyectado para 2026, 2027 y 2028?"))


## Mini-eval: 3 preguntas de negocio

Tres preguntas que un analista haría de verdad — una se responde con **texto corrido**, una con una **tabla** y una con un **gráfico**. Con OCR clásico, las dos últimas serían las primeras en fallar.

In [ ]:
preguntas = [
    # textual (pág. 5): un hecho descrito en el cuerpo del texto
    "¿Qué acuerdo internacional se anunció tras el cierre estadístico del Informe y cómo reaccionaron los mercados?",
    # tabla (pág. 6, tabla II.3): el dato vive dentro de una tabla
    "Según la tabla de crecimiento económico y cuenta corriente, ¿qué rango de crecimiento del PIB se proyecta para 2026?",
    # gráfico (pág. 8, gráfico II.6): la respuesta hay que LEERLA del gráfico
    "Según el gráfico de la brecha de actividad, ¿la brecha es positiva o negativa en el horizonte de proyección?",
]

if indice is None or not HAY_OPENAI:
    print("⚠️ Falta el índice (GEMINI_API_KEY) o la llave de OpenAI — esta celda se salta.")
else:
    for pregunta in preguntas:
        print(f"❓ {pregunta}")
        print(responder(pregunta))
        print("—" * 40)


## Cierre

**¿RAG visual o parsing (lección 5)?** No compiten: resuelven cosas distintas.

| Usa **RAG visual** cuando… | Usa **OCR / parsing** cuando… |
|---|---|
| La pregunta es "¿**dónde** está esto?" sobre muchos documentos | Necesitas los **datos** fuera del documento (a un Excel, una base, un ERP) |
| Tablas, gráficos, sellos y firmas cargan el significado | El contenido es texto corrido y limpio |
| No puedes permitirte un pipeline de extracción por cada formato | Debes validar campo a campo (esquemas, Pydantic — lección 4) |

**Costos** (orden de magnitud): indexar con `gemini-embedding-2` cuesta **US$0.00012 por página** — un archivo de 10.000 páginas ≈ US$1.2, una vez. Cada consulta: un embedding de texto (fracción de centavo) + una llamada a GPT-5 con una imagen (unos pocos centavos). Lo caro no es buscar: es responder.

**La frontera open-source**: el patrón nació en **ColPali** (embeddings *multi-vector* por parche de imagen, *late interaction*) y hoy lo encabezan **ColQwen** y **Nemotron ColEmbed** en el benchmark **ViDoRe**. Con `colpali-engine` y una GPU puedes montar esto mismo dentro de tu infraestructura, sin que ninguna página salga de la empresa. Y cuando las páginas se cuenten en millones, la matriz numpy se reemplaza por una base vectorial (Qdrant, Weaviate, pgvector) — el patrón no cambia.